In [57]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from keras.models import Sequential
from keras.layers import Dense, LSTM,Dropout
from keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

df=pd.read_csv("https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Month       144 non-null    object
 1   Passengers  144 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 2.4+ KB


In [4]:
df["Month"]=pd.to_datetime(df["Month"])
df.set_index("Month",inplace=True)

In [5]:
df.isnull().sum()

Passengers    0
dtype: int64

In [6]:
scaler=MinMaxScaler()

In [7]:
scale_val=scaler.fit_transform(df[["Passengers"]])

In [8]:
def create_window(data,window_size):
    x=[]
    y=[]
    for i in range(len(data)-window_size):
        x.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(x),np.array(y)

In [46]:
X,y=create_window(scale_val,20)

In [47]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,shuffle=False)

In [52]:
model=Sequential([
    LSTM(64,activation="tanh",return_sequences=True,input_shape=(12,1)),
    Dropout(0.1),
    LSTM(32,activation="tanh",return_sequences=False),
    Dense(32,activation="relu"),
    Dropout(0.1),
    Dense(1)
])

In [53]:
model.compile(optimizer="adam",loss="mse")

In [54]:
model.fit(X_train,y_train,epochs=100,validation_data=(X_test,y_test),callbacks=[EarlyStopping(monitor="val_loss",patience=3)])

Epoch 1/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 259ms/step - loss: 0.1123 - val_loss: 0.2154
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 0.0399 - val_loss: 0.0387
Epoch 3/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.0164 - val_loss: 0.0298
Epoch 4/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 0.0250 - val_loss: 0.0240
Epoch 5/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 0.0139 - val_loss: 0.0640
Epoch 6/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 0.0155 - val_loss: 0.0827
Epoch 7/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - loss: 0.0191 - val_loss: 0.0640


In [55]:
y_pred=model.predict(X_test)
y_pred=scaler.inverse_transform(y_pred)
y_test=scaler.inverse_transform(y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 678ms/step


In [58]:
mean_squared_error(y_test,y_pred)

17165.52621572436